In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import scipy.optimize as sco
import sklearn
import streamlit
import sqlite3

print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", plt.matplotlib.__version__)
print("scipy      :", sco.__version__ if hasattr(sco, '__version__') else "ok")
print("sklearn    :", sklearn.__version__)
print("sqlite3    :", sqlite3.sqlite_version)
print("streamlit  :", streamlit.__version__)
print("\n✅ All libraries imported successfully")

Matplotlib is building the font cache; this may take a moment.


pandas     : 3.0.2
numpy      : 2.4.4
matplotlib : 3.10.8
scipy      : ok
sklearn    : 1.8.0
sqlite3    : 3.45.3
streamlit  : 1.56.0

✅ All libraries imported successfully


In [6]:
import yfinance as yf

# VCB trên Yahoo Finance có suffix .VN
df = yf.download('VCB.VN', start='2024-01-01', end='2024-01-31')
print(df.head())
print("✅ yfinance working, shape:", df.shape)

[*********************100%***********************]  1 of 1 completed

Price              Close          High           Low          Open   Volume
Ticker            VCB.VN        VCB.VN        VCB.VN        VCB.VN   VCB.VN
Date                                                                       
2024-01-02  55448.113281  55514.519192  54584.848076  55049.681695  2837212
2024-01-03  56112.160156  56112.160156  54983.279077  55448.112691  2274044
2024-01-04  57041.832031  57241.045886  55780.135237  56112.160913  4088675
2024-01-05  57241.046875  57241.046875  56909.021193  57041.833017  1985958
2024-01-08  57639.472656  57639.472656  57307.450863  57307.450863  2470936
✅ yfinance working, shape: (21, 5)


In [7]:
import sqlite3
import os

conn = sqlite3.connect('../data/raw/portfolio.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Stock_Prices (
        Ticker TEXT,
        Date   TEXT,
        Open   REAL,
        High   REAL,
        Low    REAL,
        Close  REAL,
        Volume INTEGER,
        PRIMARY KEY (Ticker, Date)
    )
''')

conn.commit()
conn.close()

print("✅ SQLite connected, table Stock_Prices created")
print("📁 DB location:", os.path.abspath('../data/raw/portfolio.db'))

✅ SQLite connected, table Stock_Prices created
📁 DB location: D:\Projects\vn-portfolio-optimizer\data\raw\portfolio.db


In [8]:
import subprocess
subprocess.run(['pip', 'install', 'yfinance'], capture_output=True)
print("✅ yfinance installed")

✅ yfinance installed


In [9]:
import yfinance as yf

import pandas as pd

# Download với auto_adjust=False để lấy giá gốc

df_raw = yf.download('VCB.VN', 

                      start='2024-01-01', 

                      end='2024-01-31',

                      auto_adjust=False,

                      progress=False)

# Flatten multi-level columns nếu có

df_raw.columns = [col[0] if isinstance(col, tuple) else col 

                  for col in df_raw.columns]

print("Columns:", df_raw.columns.tolist())

print("\nGiá Close VCB tháng 1/2024:")

print(df_raw['Close'].head(10))

print("\nShape:", df_raw.shape)

Columns: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']

Giá Close VCB tháng 1/2024:
Date
2024-01-02    55852.843750
2024-01-03    56521.738281
2024-01-04    57458.195312
2024-01-05    57658.863281
2024-01-08    58060.199219
2024-01-09    58729.097656
2024-01-10    59866.222656
2024-01-11    59732.441406
2024-01-12    59331.105469
2024-01-15    60200.667969
Name: Close, dtype: float64

Shape: (21, 6)


In [10]:
import sys
sys.path.append('../src')
from data_loader import update_db, load_from_db, get_all_tickers

# Chạy full ETL pipeline
update_db(start='2021-01-01')

✅ Table Stock_Prices ready

📥 Fetching 10 tickers from 2021-01-01 to 2026-04-03

  VCB    → 1306 rows inserted
  VNM    →  665 rows inserted
  HPG    → 1306 rows inserted
  FPT    → 1306 rows inserted
  MWG    →  665 rows inserted
  VIC    → 1306 rows inserted
  GAS    → 1306 rows inserted
  BID    → 1306 rows inserted
  CTG    → 1306 rows inserted
  TCB    → 1303 rows inserted

✅ Done. Tickers in DB: ['BID', 'CTG', 'FPT', 'GAS', 'HPG', 'MWG', 'TCB', 'VCB', 'VIC', 'VNM']
